In [1]:
# Step 1: Import necessary libraries
import pandas as pd
import numpy as np
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_absolute_error
import matplotlib.pyplot as plt


In [2]:
def load_data(file_path):
    # Load the dataset
    df = pd.read_csv(file_path)

    # Convert transaction_date to datetime format
    df['transaction_date'] = pd.to_datetime(df['transaction_date'])

    # Set transaction_date as the index
    df.set_index('transaction_date', inplace=True)

    # Sort data by date
    df = df.sort_index()

    # Select relevant columns: quantity (target), unit_price, discount_applied, product_stock, product_category, day_of_week, season
    features = ['discount_applied', 'unit_price', 'product_stock', 'product_category', 'day_of_week', 'season', 'product_id']
    df = df[['quantity'] + features]


    return df


In [3]:
data=load_data("/Users/shivanshmahajan/Desktop/MINOR/retail_data.csv")

In [5]:
data.to_csv("1.csv")

In [5]:
# Step 3: Train-test split
def train_test_split(data, test_size=0.2):
    # Split data into train and test sets (80% train, 20% test)
    split_index = int(len(data) * (1 - test_size))
    train_data = data.iloc[:split_index]
    test_data = data.iloc[split_index:]
    
    return train_data, test_data

# Step 4: Build the ARIMA model
def build_arima_model(train_data, exog_features):
    # Fit ARIMA model (order (p, d, q)) with exogenous features for each product
    model = ARIMA(train_data['quantity'], order=(5, 1, 0), exog=train_data[exog_features])
    model_fit = model.fit()

    return model_fit


In [6]:
# Step 5: Forecast sales
def forecast_sales(model_fit, steps, exog_future):
    # Forecast sales for the future 'steps' time steps with exogenous variables
    forecast = model_fit.forecast(steps=steps, exog=exog_future)
    return forecast

# Step 6: Evaluate the model
def evaluate_model(test_data, predicted_sales):
    # Compare predicted sales with actual sales in the test set
    mae = mean_absolute_error(test_data['quantity'], predicted_sales)
    print(f"Mean Absolute Error (MAE): {mae}")

    # Plot actual vs predicted sales
    plt.figure(figsize=(10,6))
    plt.plot(test_data.index, test_data['quantity'], label="Actual Sales")
    plt.plot(test_data.index, predicted_sales, label="Predicted Sales", linestyle='--')
    plt.title("Actual vs Predicted Sales")
    plt.xlabel("Date")
    plt.ylabel("Sales Quantity")
    plt.legend()
    plt.show()


In [11]:
# Assume df is your dataset with 100,000 rows
file_path = 'retail_data.csv'  # <-- Replace this with the path to your dataset





In [25]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import LabelEncoder

# Function to load the dataset
def load_data(file_path):
    # Load the dataset
    df = pd.read_csv(file_path)

    # Convert transaction_date to datetime format
    df['transaction_date'] = pd.to_datetime(df['transaction_date'])

    # Set transaction_date as the index
    df.set_index('transaction_date', inplace=True)

    # Sort data by date
    df = df.sort_index()

    return df

In [26]:
data=load_data(file_path)

In [27]:
data

,customer_id,age,gender,income_bracket,loyalty_program,membership_years,churned,marital_status,number_of_children,education_level,...,distance_to_store,holiday_season,season,weekend,customer_support_calls,email_subscriptions,app_usage,website_visits,social_media_engagement,days_since_last_purchase
transaction_date,,,,,,,,,,,,,,,,,,,,,
2020-01-01 00:00:21,124876,20,Male,High,Yes,8,No,Divorced,2,Master's,...,63.84,Yes,Spring,Yes,1,Yes,Medium,8,Low,43
2020-01-01 00:00:59,762000,71,Male,High,Yes,4,No,Divorced,3,Bachelor's,...,56.70,Yes,Summer,No,17,No,Medium,26,Medium,120
2020-01-01 00:02:01,234082,54,Male,Medium,Yes,7,Yes,Divorced,0,PhD,...,12.45,No,Spring,Yes,18,No,Medium,75,Low,140
2020-01-01 00:03:11,552671,44,Female,Low,No,6,No,Divorced,3,Master's,...,62.49,Yes,Summer,Yes,6,No,Low,95,Medium,134
2020-01-01 00:03:22,41044,26,Female,High,Yes,0,No,Married,0,Bachelor's,...,17.11,No,Fall,No,3,No,Medium,34,Medium,11
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2021-12-31 23:55:12,301958,33,Other,Medium,Yes,3,Yes,Divorced,3,High School,...,3.59,No,Summer,No,10,No,Medium,10,Medium,275
2021-12-31 23:55:15,741035,72,Other,Low,No,5,No,Single,1,PhD,...,68.79,No,Spring,No,7,Yes,High,24,Low,275
2021-12-31 23:56:21,100823,19,Male,Low,No,9,No,Divorced,4,PhD,...,63.64,No,Fall,Yes,13,No,Medium,98,Medium,80


In [28]:
# Function for feature engineering
def feature_engineering(df):
    # Select relevant columns: quantity (target), unit_price, discount_applied, product_stock, product_category, day_of_week, season
    features = ['discount_applied', 'unit_price', 'product_stock', 'product_category', 'day_of_week', 'season', 'product_id']
    df = df[['quantity'] + features]

    # Define categorical features and numeric features
    categorical_features = ['day_of_week', 'season', 'product_category', 'product_id']
    numeric_features = ['discount_applied', 'unit_price', 'product_stock']

    # Create a ColumnTransformer to apply OneHotEncoder to categorical features
    preprocessor = ColumnTransformer(
        transformers=[
            ('cat', OneHotEncoder(drop='first'), categorical_features),
            ('num', 'passthrough', numeric_features)
        ]
    )

    # Apply the transformation
    X = preprocessor.fit_transform(df.drop('quantity', axis=1))
    y = df['quantity'].values

    return X, y


In [29]:
X,y=feature_engineering(data)

In [30]:
X

<1000000x10014 sparse matrix of type '<class 'numpy.float64'>'
	with 6387083 stored elements in Compressed Sparse Row format>

In [ ]:
# Split the data into training and testing sets
train_data, test_data = train_test_split(data)

# Build the ARIMA model using training data
exog_features = [col for col in train_data.columns if col != 'quantity']
arima_model = build_arima_model(train_data, exog_features)

# Forecast sales for the test period
predicted_sales = arima_model.forecast(steps=len(test_data), exog=test_data[exog_features])

# Evaluate the model
evaluate_model(test_data, predicted_sales)

# Step 8: Forecast Future Sales (e.g., next 3 months)
future_steps = 90  # Forecast 90 days ahead
exog_future = np.tile([0.1, 49.72, 100, 1], (future_steps, 1))  # Example values for discount_applied, unit_price, product_stock, product_category
future_sales_forecast = forecast_sales(arima_model, future_steps, exog_future)

# Print the forecasted future sales
print("Forecasted Sales for next 90 days:")
print(future_sales_forecast)